# Seattle Duwamish parcel screen: sensitivity checks

Question: how much of the parcel ranking is the data, and how much is the assumptions (distance scales, usable-area scale, weights)?

The headline claim is that the parcel evidence score separates parcels. A score spread is guaranteed by construction whenever the components are not constant, so spread alone proves nothing. The checks here ask whether the *ranking* survives reasonable changes to the assumptions.

Inputs: `data/WA/parcels/seattle_duwamish_scored.csv` (written by `parcels/seattle_sample.py`).

In [1]:
import numpy as np
import pandas as pd

FT = 0.3048
ACRE_M2 = 4047.0
scored = pd.read_csv('../data/WA/parcels/seattle_duwamish_scored.csv')
parcels = scored[~scored['gated']].reset_index(drop=True)
print(len(scored), 'screened;', len(parcels), 'not gated')

602 screened; 578 not gated


## Base score, rebuilt from raw distances
The score is rebuilt here so each assumption can be changed on its own.

In [2]:
BASE_WEIGHTS = {'res_setback': 20, 'receptor': 10, 'usable_area': 15, 'transmission': 15,
                'flood': 15, 'liquefaction': 10, 'site_hazard': 15}

In [3]:
def components(res_scale_ft=500, receptor_scale_ft=500, usable_scale_acres=1.0):
    return pd.DataFrame({
        'res_setback': np.clip(parcels['res_dist_m'] / (res_scale_ft * FT), 0, 1),
        'receptor': np.clip(parcels['receptor_dist_m'] / (receptor_scale_ft * FT), 0, 1),
        'usable_area': np.clip(parcels['usable_area_m2'] / (usable_scale_acres * ACRE_M2), 0, 1),
        'transmission': parcels['transmission_score'],
        'flood': parcels['flood_score'],
        'liquefaction': parcels['liquefaction_score'],
        'site_hazard': parcels['site_hazard_score'],
    })

In [4]:
def score(weights=BASE_WEIGHTS, **scales):
    comp = components(**scales)
    total = sum(weights.values())
    return sum(comp[k] * w for k, w in weights.items()) / total

In [5]:
base = score()
print('max abs difference vs the script score:', float((base - parcels['parcel_score']).abs().max()))

max abs difference vs the script score: 6.0292809488604604e-05


## Ties at the top
The score has a ceiling, so many parcels can tie at the top. When they do, 'top 50' is partly decided by tie-breaking order, and the top-50 overlap numbers below are inflated or deflated by arbitrary ordering. The robust-candidate set at the end, which uses top-quartile membership, is less affected.

In [6]:
ceiling = base.max()
print('ceiling score:', round(ceiling, 4))
print('parcels within 0.002 of the ceiling:', int((base >= ceiling - 0.002).sum()))
print('parcels within 0.010 of the ceiling:', int((base >= ceiling - 0.010).sum()))
print('distinct scores among the top 50:', base.nlargest(50).round(4).nunique())

ceiling score: 0.9
parcels within 0.002 of the ceiling: 37
parcels within 0.010 of the ceiling: 74
distinct scores among the top 50: 20


In [7]:
full_marks = ((parcels['res_dist_m'] >= 500 * FT) & (parcels['receptor_dist_m'] >= 500 * FT)
              & (parcels['usable_area_m2'] >= ACRE_M2))
print('parcels with full marks on setback, receptor distance and usable area:', int(full_marks.sum()), 'of', len(parcels))

parcels with full marks on setback, receptor distance and usable area: 103 of 578


## Comparison helpers
Two views of ranking stability: rank correlation with the base ranking, and how many of the base top 50 stay in the top 50.

In [8]:
TOP_N = 50
base_top = set(base.nlargest(TOP_N).index)

def compare(variant):
    rank_corr = base.corr(variant, method='spearman')
    overlap = len(base_top & set(variant.nlargest(TOP_N).index))
    return rank_corr, overlap

## A. Residential setback scale
Published WA and peer references run from 100 ft (Thurston model recommendation) to 500 ft (Whitman draft) and up to 600 ft in peer counties. The earlier run used 300 m (about 984 ft).

In [9]:
rows = []
for ft in [100, 250, 500, 984]:
    r, o = compare(score(res_scale_ft=ft))
    rows.append({'res_scale_ft': ft, 'rank_corr_vs_base': round(r, 3), 'top50_overlap': o})
pd.DataFrame(rows)

,res_scale_ft,rank_corr_vs_base,top50_overlap
0,100,0.885,44
1,250,0.936,46
2,500,1.000,50
3,984,0.922,40


## B. Usable-area scale
How much usable area counts as full marks. This one has no published reference; it is a judgement about project size.

In [10]:
rows = []
for acres in [0.25, 0.5, 1.0, 2.0]:
    r, o = compare(score(usable_scale_acres=acres))
    rows.append({'usable_scale_acres': acres, 'rank_corr_vs_base': round(r, 3), 'top50_overlap': o})
pd.DataFrame(rows)

,usable_scale_acres,rank_corr_vs_base,top50_overlap
0,0.25,0.843,17
1,0.50,0.949,22
2,1.00,1.000,50
3,2.00,0.967,27


## C. Alternative weight sets

In [11]:
equal = {k: 1 for k in BASE_WEIGHTS}
safety_heavy = {'res_setback': 30, 'receptor': 15, 'usable_area': 20, 'transmission': 5,
                'flood': 10, 'liquefaction': 5, 'site_hazard': 15}
grid_heavy = {'res_setback': 10, 'receptor': 5, 'usable_area': 10, 'transmission': 40,
              'flood': 15, 'liquefaction': 5, 'site_hazard': 15}

In [12]:
rows = []
for name, w in [('equal', equal), ('safety_heavy', safety_heavy), ('grid_heavy', grid_heavy)]:
    r, o = compare(score(weights=w))
    rows.append({'weights': name, 'rank_corr_vs_base': round(r, 3), 'top50_overlap': o})
pd.DataFrame(rows)

,weights,rank_corr_vs_base,top50_overlap
0,equal,0.972,48
1,safety_heavy,0.992,48
2,grid_heavy,0.977,46


## D. Leave one component out
A component whose removal barely moves the ranking is not carrying information. A component whose removal moves it a lot is carrying the ranking.

In [13]:
rows = []
for dropped in BASE_WEIGHTS:
    w = {k: v for k, v in BASE_WEIGHTS.items() if k != dropped}
    r, o = compare(score(weights=w))
    rows.append({'dropped': dropped, 'rank_corr_vs_base': round(r, 3), 'top50_overlap': o})
pd.DataFrame(rows).sort_values('rank_corr_vs_base')

,dropped,rank_corr_vs_base,top50_overlap
2,usable_area,0.461,10
0,res_setback,0.729,40
6,site_hazard,0.922,45
4,flood,0.992,35
1,receptor,0.994,48
5,liquefaction,0.997,50
3,transmission,0.998,38


## E. Random weight perturbation
300 draws where every weight is multiplied by a random factor between 0.5 and 1.5.

In [14]:
rng = np.random.default_rng(42)
draws = []
for _ in range(300):
    w = {k: v * rng.uniform(0.5, 1.5) for k, v in BASE_WEIGHTS.items()}
    draws.append(score(weights=w))
perturbed = pd.concat(draws, axis=1)

In [15]:
stats = [compare(perturbed[c]) for c in perturbed.columns]
rank_corrs = pd.Series([s[0] for s in stats])
overlaps = pd.Series([s[1] for s in stats])
pd.DataFrame({'rank_corr_vs_base': rank_corrs.describe().round(3), 'top50_overlap': overlaps.describe().round(1)})

,rank_corr_vs_base,top50_overlap
count,300.000,300.0
mean,0.973,47.8
std,0.022,2.4
min,0.880,38.0
25%,0.964,48.0
50%,0.979,49.0
75%,0.989,49.0
max,0.999,50.0


## F. Robust candidates
A parcel is robust if it lands in the top quartile of non-gated parcels in at least 80 percent of all variants above (scale changes, weight sets, and the random draws). These are the parcels whose standing does not depend on the assumptions.

In [16]:
variants = [score(res_scale_ft=ft) for ft in [100, 250, 984]]
variants += [score(usable_scale_acres=a) for a in [0.25, 0.5, 2.0]]
variants += [score(weights=w) for w in [equal, safety_heavy, grid_heavy]]
variants += list(draws)
variants.append(base)
all_variants = pd.concat(variants, axis=1)
print(all_variants.shape[1], 'variants')

310 variants


In [17]:
top_quartile = all_variants.apply(lambda col: col >= col.quantile(0.75))
share_in_top_quartile = top_quartile.mean(axis=1)
parcels['robust_share'] = share_in_top_quartile
parcels['robust'] = share_in_top_quartile >= 0.8
print('robust parcels:', int(parcels['robust'].sum()), 'of', len(parcels))
print('in top quartile in at least one variant:', int((share_in_top_quartile > 0).sum()))

robust parcels: 133 of 578
in top quartile in at least one variant: 275


In [18]:
cols = ['PIN', 'ADDR_FULL', 'zoning', 'PREUSE_DESC', 'acres', 'usable_area_m2', 'res_dist_m', 'parcel_score', 'robust_share']
robust = parcels[parcels['robust']].sort_values('parcel_score', ascending=False)
robust[cols].round(3).head(15)

,PIN,ADDR_FULL,zoning,PREUSE_DESC,acres,usable_area_m2,res_dist_m,parcel_score,robust_share
163,7666205350,3194 OCCIDENTAL AVE S,MML U/85,Vacant(Industrial) ...,1.440,4684.050,912.634,0.9,1.0
142,7666204355,NaN,II U/160,Vacant(Industrial) ...,1.580,4333.780,531.170,0.9,1.0
134,7666204200,2960 4TH AVE S,II U/160,Warehouse ...,2.479,4964.425,596.417,0.9,1.0
121,7666204070,NaN,MML U/85,Parking(Assoc) ...,3.984,14278.038,571.035,0.9,1.0
31,7666700515,NaN,MML U/85,Vacant(Industrial) ...,11.006,41352.949,641.509,0.9,1.0
442,2617650020,4401 4TH AVE S,MML U/85,Retail(Discount) ...,14.659,37672.711,580.074,0.9,1.0
215,7666207536,4050 E MARGINAL WAY S,MML U/85,Warehouse ...,18.366,28623.580,996.243,0.9,1.0
174,7666205610,3453 4TH AVE S,MML U/85,Service Building ...,1.538,4589.564,694.029,0.9,1.0
181,7666205845,3422 1ST AVE S,MML U/85,Industrial(Heavy) ...,2.041,5224.154,903.580,0.9,1.0
145,7666204430,2230 4TH AVE S,II U/160,Retail Store ...,3.090,5399.702,585.970,0.9,1.0


In [19]:
print('base top 50 that are robust:', int(parcels.loc[parcels.index.isin(base_top), 'robust'].sum()), 'of', TOP_N)
robust.to_csv('../data/WA/parcels/seattle_duwamish_robust.csv', index=False)

base top 50 that are robust: 50 of 50
